In [1]:
# Processing data and configurations

import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.ops import cascaded_union
import os


def preprocess_rean(xdf):
    xdf = xdf.rename({'lon':'longitude','lat':'latitude'})
    """Adjust longitude bounds and round coordinates."""
    xdf['longitude'] = np.round(xdf.longitude, 3)
    xdf['latitude'] = np.round(xdf.latitude, 3)
    xdf = xdf.drop('time_bounds')
    
    if xdf.longitude.values.max() >= 300:
        xdf = roll_xdf(xdf)

    if exp == 'historical':
        xdf = xdf.sel(time=slice('1985','2014'))
    else:
        xdf = xdf.sel(time=slice('2015','2100'))
    xdf = xdf.mean(['member_id','dcpp_init_year']).drop('height')
    return xdf

def roll_xdf(xdf):
    """Roll longitude from 0-360 to -180-180."""
    xdf = xdf.assign_coords(longitude=(((xdf.longitude + 180) % 360) - 180))
    xdf = xdf.sortby(xdf.longitude)
    return xdf


def get_geometries(cut_on, shapefile=None):
    """Retrieve geometries for given region or bounding box coordinates."""
    if cut_on == 'europe':
        geometries = [-25, 32, 48.5, 75]
    elif cut_on == 'SADC':
        geometries = [2, -37, 60, 7]
    elif isinstance(cut_on, list):
        geometries = cut_on
    else:   
        if not shapefile:
            geometries = get_boundaries(cut_on.capitalize(), continent=False)
        else:
            geometries = gpd.read_file(shapefile)
            geometries = geometries[geometries['NAME_EN'] == cut_on]['geometry']
    return geometries

def get_boundaries(name, continent=False):
    """Retrieve boundary geometries by name."""
    world_path = '../world_map/'
    world = gpd.read_file(world_path)
    
    if continent:
        world = world.query('REGION_UN == @name')
        boundary = gpd.GeoSeries(cascaded_union(world['geometry']))
    elif name in world['NAME_EN'].values:
        region_mask = world[world['NAME_EN'] == name]
        polygons = region_mask['geometry']
        boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)
    return boundary

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [2]:
from datetime import datetime
import glob

def load_data(data_path):
    """Load river discharge dataset based on specified variable."""
    #full_path = f"{data_path}/"
    #files = sorted(glob.glob(full_path + 'era5land_surface_solar_radiation_*.nc'))  #?
    #print(files)
    try:
        xdf = xr.open_mfdataset(data_path+'/*.nc', preprocess=preprocess_rean)
    except IOError as e:
        raise IOError(f"Files not found at {data_path}. Error: {e}")
    return xdf


def include_time_coord(ds):
    """Ensure time coordinate inclusion."""
    if 'time' not in ds.coords:
        ds = ds.assign_coords(time=('time', pd.date_range(datetime.now(), periods=ds.dims['time'])))
    return ds

def normalize_naming(xdf):
    """Standardize variable names as required."""
    # Placeholder normalization, as naming conventions may vary
    return xdf.rename({'var_name_placeholder': 'variable_standard_name'}) 

def aggregate_daymean(xdf, var_name):
    """Compute daily mean for given variable."""
    #xdf = xdf.rename({'valid_time':'time'})
    xdf_daymean = xdf[var_name].resample(time='1D').mean()
    return xdf_daymean

def cut_xdf(xdf, geometries):
    
    xdf = xdf.sortby('latitude')
    """Subset xdf using provided geometries."""
    # Assuming geometries are bounding box [lon_min, lat_min, lon_max, lat_max]
    # Assuming 'geometries' is a DataFrame with bounding box columns
    lon_min = geometries.bounds['minx'].values[0]  # Extract minimum longitude
    lat_min = geometries.bounds['miny'].values[0]  # Extract minimum latitude
    lon_max = geometries.bounds['maxx'].values[0]  # Extract maximum longitude
    lat_max = geometries.bounds['maxy'].values[0]  # Extract maximum latitude
    
    xdf_cut = xdf.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))
    return xdf_cut

In [3]:
def main(variable, model, experiment, start_date, end_date, loading_area, aggregation_mode, overwrite, data_path, output_path):
    # Step 1: Define date range and load data
    xdf = load_data(data_path=data_path)
    
    # Step 2: Apply geographic masking if specified
    if loading_area:
        geometries = get_geometries(loading_area)
        xdf = cut_xdf(xdf, geometries)
        print(f"Data masked to the specified loading area: {loading_area}")

    # Step 3: Aggregate data by daily mean if required
    if aggregation_mode == 'daymean':
        xdf = aggregate_daymean(xdf, var_name=variable)
        print("Data aggregated by daily mean.")

    # Step 4: Save output if overwrite is enabled
    output_filename = f"{output_path}/{variable}_{start_date}_{end_date}_processed.nc"
    if overwrite or not os.path.exists(output_filename):
        os.makedirs(output_path, exist_ok=True)
        print(xdf)
        xdf.to_netcdf(output_filename)
        print(f"Data written to {output_filename}")
    else:
        print(f"File already exists and overwrite is set to False: {output_filename}")

In [4]:
variables = ['uas','vas']
model = 'CNRM-ESM2-1'
exps = ['historical', 'ssp370', 'ssp585']
country = 'cuba' ## FILL WITH THE COUNTRY NAME e.g. croatia, cuba

# Processing data and configurations
shapefile_path = '../world_map/ne_50m_admin_0_countries.shp'
geometries = get_geometries(f'{country.capitalize()}', shapefile_path)

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


In [7]:
for var in variables:
    for exp in exps:
        #data_path = f'../data/cds_downloads/{exp}/{model}/' ## FILL WITH THE PATH of ERA5LAND data
        output_path = f'../data/processed_data/{country}/{exp}/{model}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
        output_dir = os.path.dirname(output_path)
        os.makedirs(output_dir, exist_ok=True)
        if exp == 'historical':
            start = '1985-01'
            end = '2014-12'
        else:
            start = '2015-01'
            end = '2100-12'
    
        # Parameters (Example Usage)
        main(
            variable         = var,
            model            = model,
            experiment       = exp,
            start_date       = start,
            end_date         = end,
            loading_area     = country.capitalize(),
            aggregation_mode = None,
            data_path        = data_path,
            output_path      = output_path,
            overwrite        = True)

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/tmp/ipykernel_1023346/4056308480.py:62: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)


Data masked to the specified loading area: Cuba
<xarray.Dataset>
Dimensions:    (latitude: 3, longitude: 8, time: 10957)
Coordinates:
  * latitude   (latitude) float64 20.31 21.71 23.11
  * longitude  (longitude) float64 -84.38 -82.97 -81.56 ... -77.34 -75.94 -74.53
  * time       (time) datetime64[ns] 1985-01-01T12:00:00 ... 2014-12-31T12:00:00
Data variables:
    uas        (time, latitude, longitude) float32 dask.array<chunksize=(10957, 3, 8), meta=np.ndarray>
Data written to ../data/processed_data/cuba/historical/CNRM-ESM2-1//uas_1985-01_2014-12_processed.nc


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/tmp/ipykernel_1023346/4056308480.py:62: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)


Data masked to the specified loading area: Cuba
<xarray.Dataset>
Dimensions:    (latitude: 3, longitude: 8, time: 31411)
Coordinates:
  * latitude   (latitude) float64 20.31 21.71 23.11
  * longitude  (longitude) float64 -84.38 -82.97 -81.56 ... -77.34 -75.94 -74.53
  * time       (time) datetime64[ns] 2015-01-01T12:00:00 ... 2100-12-31T12:00:00
Data variables:
    uas        (time, latitude, longitude) float32 dask.array<chunksize=(31411, 3, 8), meta=np.ndarray>
Data written to ../data/processed_data/cuba/ssp370/CNRM-ESM2-1//uas_2015-01_2100-12_processed.nc
Data masked to the specified loading area: Cuba
<xarray.Dataset>
Dimensions:    (latitude: 3, longitude: 8, time: 31411)
Coordinates:
  * latitude   (latitude) float64 20.31 21.71 23.11
  * longitude  (longitude) float64 -84.38 -82.97 -81.56 ... -77.34 -75.94 -74.53
  * time       (time) datetime64[ns] 2015-01-01T12:00:00 ... 2100-12-31T12:00:00
Data variables:
    uas        (time, latitude, longitude) float32 dask.array<chunksize

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/tmp/ipykernel_1023346/4056308480.py:62: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)


...

Data masked to the specified loading area: Cuba
<xarray.Dataset>
Dimensions:    (latitude: 3, longitude: 8, time: 10957)
Coordinates:
  * latitude   (latitude) float64 20.31 21.71 23.11
  * longitude  (longitude) float64 -84.38 -82.97 -81.56 ... -77.34 -75.94 -74.53
  * time       (time) datetime64[ns] 1985-01-01T12:00:00 ... 2014-12-31T12:00:00
Data variables:
    vas        (time, latitude, longitude) float32 dask.array<chunksize=(10957, 3, 8), meta=np.ndarray>
Data written to ../data/processed_data/cuba/historical/CNRM-ESM2-1//vas_1985-01_2014-12_processed.nc


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/tmp/ipykernel_1023346/4056308480.py:62: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)


Data masked to the specified loading area: Cuba
<xarray.Dataset>
Dimensions:    (latitude: 3, longitude: 8, time: 31411)
Coordinates:
  * latitude   (latitude) float64 20.31 21.71 23.11
  * longitude  (longitude) float64 -84.38 -82.97 -81.56 ... -77.34 -75.94 -74.53
  * time       (time) datetime64[ns] 2015-01-01T12:00:00 ... 2100-12-31T12:00:00
Data variables:
    vas        (time, latitude, longitude) float32 dask.array<chunksize=(31411, 3, 8), meta=np.ndarray>
Data written to ../data/processed_data/cuba/ssp370/CNRM-ESM2-1//vas_2015-01_2100-12_processed.nc


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/tmp/ipykernel_1023346/4056308480.py:62: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)


Data masked to the specified loading area: Cuba
<xarray.Dataset>
Dimensions:    (latitude: 3, longitude: 8, time: 31411)
Coordinates:
  * latitude   (latitude) float64 20.31 21.71 23.11
  * longitude  (longitude) float64 -84.38 -82.97 -81.56 ... -77.34 -75.94 -74.53
  * time       (time) datetime64[ns] 2015-01-01T12:00:00 ... 2100-12-31T12:00:00
Data variables:
    vas        (time, latitude, longitude) float32 dask.array<chunksize=(31411, 3, 8), meta=np.ndarray>
Data written to ../data/processed_data/cuba/ssp585/CNRM-ESM2-1//vas_2015-01_2100-12_processed.nc
